In [4]:
import numpy as np
import tensorflow as tf
import pandas as pd
from PIL import Image
from tqdm import tqdm

Фіксимо помилку при завантажені моделі (зроблено для застарілих версіх TensorFlow)

In [5]:
import h5py

f = h5py.File("cards/14card types-14-(200 X 200)-94.61.h5", mode="r+")
model_config_string = f.attrs.get("model_config")

if model_config_string.find('"groups": 1,') != -1:
    model_config_string = model_config_string.replace('"groups": 1,', '')
f.attrs.modify('model_config', model_config_string)
f.flush()

model_config_string = f.attrs.get("model_config")

assert model_config_string.find('"groups": 1,') == -1

In [6]:
model = tf.keras.models.load_model('cards/14card types-14-(200 X 200)-94.61.h5')

C:\LData\LabsKPI13\.venv\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [8]:
#model.summary()
# На випадок як забажаєте глянути з чого зроблена модель

In [5]:
# Feature extractor
feature_layer_name = 'dense_6'
feature_extractor = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.get_layer(feature_layer_name).output
)


In [34]:
cards = pd.read_csv('cards/processed_cards.csv')

In [31]:
cards['features'] = [None] * len(cards)  # Initialize an empty column for features
cards['extracted_label'] = [None] * len(cards) # Initialize an empty column for extracted labels


In [35]:
cards['features'] = cards['features'].apply(lambda x: np.fromstring(x[1:-1], sep=' ') if pd.notnull(x) else None)
cards['features'].to_list()[0]

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.5492758 , 0.        ,
       0.        , 0.        , 0.        , 0.9262029 , 0.        ,
       0.        , 0.        , 0.09862591, 0.        , 0.        ,
       0.        , 2.7218359 , 0.        , 1.8312975 , 2.5269432 ,
       0.        , 0.        , 0.        , 0.19412917, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.04645473,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 1.6116221 , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [36]:
cards['features'].to_list()[0][0]

np.float64(0.0)

In [40]:
for i, row in tqdm(cards.iterrows()):
    image_path = row['filepaths']
    image_path = f'cards/{image_path}'
    if row['features'] is not None:
        continue  # Skip if features are already extracted
    try:
        image = Image.open(image_path).convert('RGB')
    except:
        continue
    image = image.resize((200, 200))
    features = feature_extractor(tf.expand_dims(tf.convert_to_tensor(image), axis=0))
    extracted_label = model.predict(tf.expand_dims(tf.convert_to_tensor(image), axis=0)).argmax()
    cards.at[i, 'features'] = features.numpy().flatten()
    cards.at[i, 'extracted_label'] = extracted_label

2717it [00:00, 9898.39it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step


2717it [00:11, 9898.39it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step


3032it [00:11, 168.18it/s] 

...

7098it [1:01:25,  1.06it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step


7099it [1:01:26,  1.04it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step


7115it [1:01:27,  1.93it/s]


In [46]:
cards['card type encoded'] = cards['card type'].astype('category').cat.codes

In [45]:
cards

,class index,filepaths,labels,card type,data set,suit,labels_encoded,type,type suit,final_label,features,extracted_label,card type encoded
0,0,train/ace of clubs/001.jpg,ace of clubs,ace,train,clubs,0,number,number clubs,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0
1,0,train/ace of clubs/002.jpg,ace of clubs,ace,train,clubs,0,number,number clubs,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.02084561, 0.552028...",0.0,0
2,0,train/ace of clubs/003.jpg,ace of clubs,ace,train,clubs,0,number,number clubs,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.091402613, 0....",0.0,0
3,0,train/ace of clubs/004.jpg,ace of clubs,ace,train,clubs,0,number,number clubs,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.02104898, 0.0...",0.0,0
4,0,train/ace of clubs/005.jpg,ace of clubs,ace,train,clubs,0,number,number clubs,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7110,52,valid/two of spades/1.jpg,two of spades,two,valid,spades,52,number,number spades,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",12.0,12
7111,52,valid/two of spades/2.jpg,two of spades,two,valid,spades,52,number,number spades,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",12.0,12
7112,52,valid/two of spades/3.jpg,two of spades,two,valid,spades,52,number,number spades,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",12.0,12
7113,52,valid/two of spades/4.jpg,two of spades,two,valid,spades,52,number,number spades,number,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",12.0,12


In [41]:
cards.to_csv('cards/processed_cards.csv', index=False)

In [43]:
cards_no_features = cards[cards['features'].isnull()]
cards = cards[cards['features'].notnull()]

In [46]:
cards_no_features

,class index,filepaths,labels,card type,data set,suit,labels_encoded,type,type suit,final_label,features,extracted_label,card type encoded
120,0,train/ace of clubs/output,ace of clubs,ace,train,clubs,0,number,number clubs,number,None,NaN,0


In [47]:
cards.to_csv('cards/featured_cards.csv', index=False)